In [1]:
import torch
import torchvision.transforms as transforms
from torchvision import models
from flask import Flask, request, jsonify
from PIL import Image
import io

c:\Users\DELL\anaconda3-updated\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
# Initialize Flask app
app = Flask(__name__)

In [3]:
# Load the pretrained ResNet-50 model
model = models.resnet50(pretrained=False)  # Use False since we load a custom model
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 14)  # Modify based on your number of classes
model.load_state_dict(torch.load("ResNet-50_model.pth", map_location=torch.device("cpu")))
model.eval()  # Set model to evaluation mode

c:\Users\DELL\anaconda3-updated\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\DELL\anaconda3-updated\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [4]:
# Define image transformations (match the ones used during training)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet-50 requires 224x224 images
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [7]:
@app.route('/')
def home():
    return "Welcome to the Land Type Classification API! Use /predict to send an image."


In [24]:
if "predict" in app.view_functions:
    app.view_functions.pop("predict")

In [25]:
@app.route("/predict", methods=["POST"])
def predict():  # Renamed function
    try:   
        file = request.files["file"]
        image = Image.open(io.BytesIO(file.read())).convert("RGB")

        image = transform(image).unsqueeze(0)  

        with torch.no_grad():
            output = model(image)
            probabilities = torch.nn.functional.softmax(output, dim=1)[0]
            predicted_class = torch.argmax(probabilities).item()
            confidence = round(probabilities[predicted_class].item() * 100, 2)  # Ensure confidence is calculated

        return jsonify({"prediction": predicted_class, "confidence": confidence})  # ✅ Ensure confidence is included

    except Exception as e:
        return jsonify({'error': str(e)})

 

In [26]:
# Run the Flask app
if __name__ == '__main__':
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1